In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



I0000 00:00:1786382829.776709  186578 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786382833.236652  186578 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# new lag features with baseling rolling features

In [3]:
# df = pd.read_csv("/kaggle/input/datasets/logeshm0324/pjme-dataset/df_for_EDA.csv")
df = pd.read_csv("../Dataset/df_for_EDA.csv")



df["Datetime"] = pd.to_datetime(df["Datetime"])

df["Hour"] = df["Datetime"].dt.hour

df["Day"] = df["Datetime"].dt.day

df["DayOfWeek"] = df["Datetime"].dt.dayofweek

df["Week"] = df["Datetime"].dt.isocalendar().week.astype(int)

df["Month"] = df["Datetime"].dt.month

df["Year"] = df["Datetime"].dt.year

df["IsWeekend"] = ( df["DayOfWeek"] >= 5 ).astype(int)

# Lag 

df["Lag_1"] = df["PJME_MW"].shift(1)
df["Lag_2"] = df["PJME_MW"].shift(2)
df["Lag_3"] = df["PJME_MW"].shift(3)
df["Lag_6"] = df["PJME_MW"].shift(6)
df["Lag_12"] = df["PJME_MW"].shift(12)
df["Lag_24"] = df["PJME_MW"].shift(24)
df["Lag_48"] = df["PJME_MW"].shift(48)
df["Lag_72"] = df["PJME_MW"].shift(72)
df["Lag_168"] = df["PJME_MW"].shift(168)

# Rolling mean and std

df["RollingMean_24"] = (
    df["PJME_MW"]
    .rolling(24)
    .mean()
)


df["RollingMean_168"] = (
    df["PJME_MW"]
    .rolling(168)
    .mean()
)


df["RollingStd_24"] = (
    df["PJME_MW"]
    .rolling(24)
    .std()
)

In [4]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,...,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24,Lag_2,Lag_3,Lag_6,Lag_12,Lag_48,Lag_72
0,1998-12-24 01:00:00,27213.0,1,3,12,24,52,1998,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1998-12-24 02:00:00,25643.0,2,3,12,24,52,1998,0,27213.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1998-12-24 03:00:00,24907.0,3,3,12,24,52,1998,0,25643.0,...,NaN,NaN,NaN,NaN,27213.0,NaN,NaN,NaN,NaN,NaN
3,1998-12-24 04:00:00,24721.0,4,3,12,24,52,1998,0,24907.0,...,NaN,NaN,NaN,NaN,25643.0,27213.0,NaN,NaN,NaN,NaN
4,1998-12-24 05:00:00,25144.0,5,3,12,24,52,1998,0,24721.0,...,NaN,NaN,NaN,NaN,24907.0,25643.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145193,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,...,42112.0,40343.500000,41856.327381,2295.270146,44147.0,41213.0,38726.0,40154.0,43308.0,45896.0
145194,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,...,40797.0,40282.750000,41873.910714,2177.148998,44343.0,44147.0,38737.0,40309.0,42440.0,45377.0
145195,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,...,38819.0,40230.208333,41895.238095,2106.081917,44284.0,44343.0,39337.0,39884.0,40661.0,44092.0
145196,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,...,36287.0,40171.166667,41918.315476,2086.336995,43751.0,44284.0,41213.0,39544.0,38207.0,42257.0


In [5]:
df.isna().sum()

Datetime             0
PJME_MW              0
Hour                 0
DayOfWeek            0
Month                0
Day                  0
Week                 0
Year                 0
IsWeekend            0
Lag_1                1
Lag_24              24
Lag_168            168
RollingMean_24      23
RollingMean_168    167
RollingStd_24       23
Lag_2                2
Lag_3                3
Lag_6                6
Lag_12              12
Lag_48              48
Lag_72              72
dtype: int64

In [6]:
missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().mean() * 100
)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

missing_summary

,Missing Count,Missing Percentage
Datetime,0,0.000000
PJME_MW,0,0.000000
Hour,0,0.000000
DayOfWeek,0,0.000000
Month,0,0.000000
Day,0,0.000000
Week,0,0.000000
Year,0,0.000000
IsWeekend,0,0.000000
Lag_1,1,0.000689


In [7]:
df = df.dropna().reset_index(drop=True)

In [8]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,...,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24,Lag_2,Lag_3,Lag_6,Lag_12,Lag_48,Lag_72
0,1998-12-17 01:00:00,29971.0,1,3,12,17,51,1998,0,32323.0,...,27213.0,35721.375000,31485.982143,3667.762970,35430.0,38234.0,40810.0,35867.0,29752.0,26437.0
1,1998-12-17 02:00:00,29046.0,2,3,12,17,51,1998,0,29971.0,...,25643.0,35676.666667,31506.238095,3744.754089,32323.0,35430.0,40317.0,35318.0,28489.0,24978.0
2,1998-12-17 03:00:00,28653.0,3,3,12,17,51,1998,0,29046.0,...,24907.0,35625.958333,31528.535714,3833.978618,29971.0,32323.0,39738.0,34817.0,27886.0,24353.0
3,1998-12-17 04:00:00,28774.0,4,3,12,17,51,1998,0,28653.0,...,24721.0,35573.875000,31552.660714,3920.893358,29046.0,29971.0,38234.0,34906.0,27712.0,24029.0
4,1998-12-17 05:00:00,29453.0,5,3,12,17,51,1998,0,28774.0,...,25144.0,35520.541667,31578.309524,3997.559484,28653.0,29046.0,35430.0,36661.0,28217.0,24363.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145025,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,...,42112.0,40343.500000,41856.327381,2295.270146,44147.0,41213.0,38726.0,40154.0,43308.0,45896.0
145026,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,...,40797.0,40282.750000,41873.910714,2177.148998,44343.0,44147.0,38737.0,40309.0,42440.0,45377.0
145027,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,...,38819.0,40230.208333,41895.238095,2106.081917,44284.0,44343.0,39337.0,39884.0,40661.0,44092.0
145028,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,...,36287.0,40171.166667,41918.315476,2086.336995,43751.0,44284.0,41213.0,39544.0,38207.0,42257.0


In [9]:
df.isna().sum()

Datetime           0
PJME_MW            0
Hour               0
DayOfWeek          0
Month              0
Day                0
Week               0
Year               0
IsWeekend          0
Lag_1              0
Lag_24             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
RollingStd_24      0
Lag_2              0
Lag_3              0
Lag_6              0
Lag_12             0
Lag_48             0
Lag_72             0
dtype: int64

In [10]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_48",
    "Lag_72"
]

target = "PJME_MW"

X = df[features]
y = df[target]

In [11]:
train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)

X_train = X.iloc[:train_size]
X_val = X.iloc[train_size:train_size + val_size]
X_test = X.iloc[train_size + val_size:]

y_train = y.iloc[:train_size]
y_val = y.iloc[train_size:train_size + val_size]
y_test = y.iloc[train_size + val_size:]

In [12]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)

X_val_scaled = X_scaler.transform(X_val)

X_test_scaled = X_scaler.transform(X_test)

In [13]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = y_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1, 1)
)

In [14]:
def create_sequences(X, y, sequence_length, forecast_horizon):

    X_sequences = []
    y_sequences = []

    for i in range(
        sequence_length,
        len(X) - forecast_horizon + 1
    ):

        X_sequences.append(
            X[i-sequence_length:i]
        )

        y_sequences.append(
            y[i:i+forecast_horizon]
        )

    return np.array(X_sequences), np.array(y_sequences)

In [ ]:
SEQUENCE_LENGTH = 48
FORECAST_HORIZON = 24


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

NameError: name 'sequence_length' is not defined

In [ ]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [ ]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=13
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/10
 810/1586 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: nan - mae: nan

KeyboardInterrupt: 

In [ ]:
bilstm_pred_48 = bilstm_model_48.predict(
    X_test_seq
)

679/679 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [ ]:
bilstm_pred_48

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)